# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [1]:
STUDENT_NAME = 'Корнілов Артем Сергійович'     # напр. 'Коваленко Іван Петрович'
LIST_NUMBER  = 10      # номер у списку групи

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 10: Луцьк, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [2]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [3]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [4]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_луцьк


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [5]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

overpass-api.de: машин 2, відповідають 2 ['162.55.144.139', '65.109.112.52']
закріплено 162.55.144.139


'162.55.144.139'

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [6]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def retry(fn, tries=4, pause=20):
    # Overpass нестабільний: повторюємо запит, а не кидаємо весь збір
    for k in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            print('спроба %d з %d не вдалась: %s' % (k, tries, type(e).__name__))
            if k == tries:
                raise
            time.sleep(pause)

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    # ----- живий запит -----
    t0, b0 = time.time(), cache_bytes()
    raw = retry(getter)
    g = clean(raw)
    g.to_parquet(path, index=False)
    g = gpd.read_parquet(path)
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return g

In [7]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

межа міста отримана, площа 40.7 км²


In [8]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# Мережа приходить двома таблицями одразу, тому їй окрема гілка.
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = retry(lambda: ox.graph_from_polygon(poly, network_type='walk'))
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

вузлів 9566 | ребер 27222


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [9]:
cost = pd.DataFrame(COST).T
display(cost)

# При першому (живому) запуску зберігаємо витрати, щоб після перезапуску
# ноутбука (коли все читається зі знімка) було з чим порівнювати.
LIVE_P = 'cost_live.json'
if any(v['спосіб'] == 'живий запит' for v in COST.values()):
    with open(LIVE_P, 'w', encoding='utf-8') as f:
        json.dump(COST, f, ensure_ascii=False, indent=1)

if os.path.exists(LIVE_P):
    with open(LIVE_P, encoding='utf-8') as f:
        live = pd.DataFrame(json.load(f)).T
    print('\n--- витрати першого, живого запуску ---')
    display(live)
    t_live = float(live['секунд'].astype(float).sum())
    t_snap = float(cost['секунд'].astype(float).sum())
    mb_saved = float(live['передано_МБ'].astype(float).sum()) - float(cost['передано_МБ'].astype(float).sum())
    print('живий запит : %.1f с, передано %.2f МБ' % (t_live, live['передано_МБ'].astype(float).sum()))
    print('знімок      : %.2f с, передано %.2f МБ' % (t_snap, cost['передано_МБ'].astype(float).sum()))
    print('читання знімка швидше у %.0f разів; мережею не передано %.2f МБ'
          % (t_live / max(t_snap, 1e-9), mb_saved))

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.028,0.338
heritage,знімок,203,0.0,0.121,0.255
stations,знімок,6,0.0,0.03,0.142
amenities,знімок,233,0.0,0.079,0.245
мережа walk,знімок,36788,0.0,1.81,0.32



--- витрати першого, живого запуску ---


,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,живий запит,1,0.038,0.028,1.6
heritage,живий запит,203,0.213,0.121,74.0
stations,живий запит,6,0.007,0.03,61.8
amenities,живий запит,233,0.083,0.079,75.1
мережа walk,живий запит,36788,5.016,1.81,83.8


живий запит : 296.3 с, передано 5.36 МБ
знімок      : 1.30 с, передано 0.00 МБ
читання знімка швидше у 228 разів; мережею не передано 5.36 МБ


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [10]:
mask = heritage['historic'].notna() | heritage['tourism'].isin(TOURISM_OK)
has_name = heritage['name'].notna()
cand = heritage[mask & has_name].reset_index(drop=True)

print('усього у шарі heritage :', len(heritage))
print('підходять за тегами    :', int(mask.sum()))
print('з них мають name       :', len(cand))
print('відкинуто              :', len(heritage) - len(cand))
print('   не підходять за тегами       :', int((~mask).sum()))
print('   підходять, але без name      :', int((mask & ~has_name).sum()))
print()
print('відкинуті за відсутністю name, за типом:')
nm = heritage[mask & ~has_name]
print(pd.concat([nm['historic'].fillna('-'), nm['tourism'].fillna('-')], axis=1)
        .value_counts().head(10).to_string())

усього у шарі heritage : 203
підходять за тегами    : 203
з них мають name       : 140
відкинуто              : 63
   не підходять за тегами       : 0
   підходять, але без name      : 63

відкинуті за відсутністю name, за типом:
historic   tourism  
building   -            34
citywalls  -             9
yes        -             8
memorial   -             5
-          artwork       4
           viewpoint     2
factory    -             1


In [11]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial           24
building           17
monument           11
yes                 8
tower               4
aircraft            2
castle              2
railway_station     1

=== tourism ===
tourism
artwork       52
museum        13
attraction     6
yes            6
gallery        3
viewpoint      2

типи геометрії: {'Point': 98, 'Polygon': 42}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [12]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

# сигнал 1: є стаття (wikidata або wikipedia)
cand['wiki'] = (cand['wikidata'].notna() | cand['wikipedia'].notna()).astype(int)

# сигнал 2: на скількох мовах названо (колонки name:*)
lang_cols = [c for c in cand.columns if c.startswith('name:')]
cand['n_lang'] = cand[lang_cols].notna().sum(axis=1)

# сигнал 3: призначений для відвідування (tourism)
VISIT = {'museum': 1.0, 'gallery': 1.0, 'attraction': 0.8, 'viewpoint': 0.6, 'artwork': 0.3}
cand['visit'] = cand['tourism'].map(VISIT).fillna(0.0)

# сигнал 4: історична природа (historic), з градацією за типом
HIST = {'castle': 1.0, 'citywalls': 1.0, 'tower': 1.0, 'ruins': 1.0,
        'monument': 0.5, 'memorial': 0.5}
cand['hist'] = cand['historic'].map(lambda v: 0.0 if pd.isna(v) else HIST.get(v, 0.6))

# сигнал 5: площа (найслабший сигнал, лише розрізняє рівні)
cand['area_pct'] = cand['area_m2'].rank(pct=True)
cand['lang_pct'] = cand['n_lang'].rank(pct=True)

W = {'wiki': 0.35, 'lang_pct': 0.25, 'visit': 0.20, 'hist': 0.15, 'area_pct': 0.05}
assert abs(sum(W.values()) - 1) < 1e-9
cand['score'] = sum(w * cand[c] for c, w in W.items())

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають name:* :', int((cand['n_lang'] > 0).sum()), '| максимум мов:', int(cand['n_lang'].max()))
print('ваги         :', W)

кандидатів   : 140
мають wiki   : 44
мають name:* : 140 | максимум мов: 14
ваги         : {'wiki': 0.35, 'lang_pct': 0.25, 'visit': 0.2, 'hist': 0.15, 'area_pct': 0.05}


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [13]:
N_SMALL, N_LARGE = 10, 30

def key(g):
    return list(zip(g['element'], g['id']))

top_score = cand.nlargest(N_LARGE, 'score').reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = cand.nlargest(N_SMALL, 'area_m2').reset_index(drop=True)
overlap   = len(set(key(top10)) & set(key(top_naive)))
print('перетин: %d з %d' % (overlap, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'за площею': list(top_naive['name'])}))

перетин: 3 з 10


,за вашим критерієм,за площею
0,Художній музей,Окільний замок
1,Волинський краєзнавчий музей,Замок Любарта
2,Музей волинської ікони,Музей військової техніки
3,Музей книги,Луцький Свято-Архангельський чоловічий монастир
4,Замок Любарта,Монастир шариток
5,В'їзна вежа,Монастир бригідок
6,Стирова вежа,Головний корпус ВНУ
7,Владича вежа,Колегіум Єзуїтів
8,Костел святих Петра і Павла,Костел святих Петра і Павла
9,Кафедральний собор,Кафедральний собор


### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [14]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

def show_pairs(top):
    pairs = close_pairs(top)
    print('пар ближче 60 м:', len(pairs))
    for i, j, d in pairs:
        print('  %5.1f м | #%d %-40s (%s) | #%d %-40s (%s)' % (
            d, i, top['name'][i][:40], top['element'][i],
            j, top['name'][j][:40], top['element'][j]))
    return pairs

pairs30 = show_pairs(top_score)

# Рішення. Пару вважаємо дублікатом, якщо (а) однакова назва або
# (б) вузол лежить усередині контуру іншого об'єкта (мітка на будівлі).
# Лише для малих будівель (< 3000 м²): у великому комплексі (замок, монастир)
# вузли всередині — це окремі об'єкти. Два перетинні контури — теж різні об'єкти.
# Тоді лишаємо запис із вищим score. Усе інше — різні об'єкти, що просто
# стоять поруч (наприклад, два пам'ятники), їх ми лишаємо обидва.
def dedupe(c):
    c = c.sort_values('score', ascending=False).reset_index(drop=True)
    metric = c.to_crs(32635)
    drop = set()
    pts = metric.geometry.representative_point()
    for i in range(len(c)):
        if i in drop:
            continue
        for j in range(i + 1, len(c)):
            if j in drop:
                continue
            if pts[i].distance(pts[j]) >= 60:
                continue
            same_name = str(c['name'][i]).strip().lower() == str(c['name'][j]).strip().lower()
            one_point = 'Point' in (metric.geometry[i].geom_type, metric.geometry[j].geom_type)
            small = min(metric.geometry[i].area, metric.geometry[j].area) == 0 and max(metric.geometry[i].area, metric.geometry[j].area) < 3000
            inside = one_point and small and metric.geometry[i].intersects(metric.geometry[j])
            if same_name or inside:
                drop.add(j)
    print('вилучені як дублікати:', [str(c['name'][k]) for k in sorted(drop)])
    return c.drop(index=sorted(drop)).reset_index(drop=True), drop

n_before = len(cand)
cand_all = cand.copy()
cand, dropped = dedupe(cand)
print('\nдублікатів вилучено з усіх кандидатів: %d (було %d, стало %d)'
      % (len(dropped), n_before, len(cand)))

top_score = cand.nlargest(N_LARGE, 'score').reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = cand.nlargest(N_SMALL, 'area_m2').reset_index(drop=True)
overlap   = len(set(key(top10)) & set(key(top_naive)))
print('після очищення: перетин двох рейтингів %d з %d' % (overlap, N_SMALL))
print('пар ближче 60 м у новому top_score:')
pairs30 = show_pairs(top_score)
display(top_score[['name', 'historic', 'tourism', 'wiki', 'n_lang', 'score']].round(3))

пар ближче 60 м: 12
    3.6 м | #0 Художній музей                           (node) | #14 Шляхетський будинок                      (way)
   16.8 м | #23 Лесина вітальня                          (node) | #28 Кликун Вогняр                            (node)
   25.5 м | #3 Музей книги                              (way) | #7 Владича вежа                             (way)
   31.6 м | #3 Музей книги                              (way) | #4 Замок Любарта                            (way)
   43.4 м | #0 Художній музей                           (node) | #5 В'їзна вежа                              (way)
   45.6 м | #5 В'їзна вежа                              (way) | #22 Дім Пузини                               (way)
   46.4 м | #5 В'їзна вежа                              (way) | #14 Шляхетський будинок                      (way)
   47.7 м | #0 Художній музей                           (node) | #4 Замок Любарта                            (way)
   50.3 м | #4 Замок Любарта                            (w

,name,historic,tourism,wiki,n_lang,score
0,Художній музей,NaN,museum,1,5,0.815
1,Волинський краєзнавчий музей,NaN,museum,1,3,0.811
2,Музей волинської ікони,NaN,museum,1,3,0.806
3,Музей книги,NaN,museum,1,3,0.802
4,Замок Любарта,castle,yes,1,14,0.800
5,В'їзна вежа,tower,NaN,1,3,0.754
6,Стирова вежа,tower,NaN,1,3,0.753
7,Владича вежа,tower,NaN,1,3,0.753
8,Костел святих Петра і Павла,building,yes,1,4,0.728
9,Кафедральний собор,yes,NaN,1,4,0.728


---
## 4. Старт: вокзал, і прив'язка точок до графа

In [15]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 6
                       name  railway       building
0                  Водограй  station            NaN
1                Промислова  station            NaN
2                     Луцьк  station            NaN
3                 Автозавод     halt            NaN
4                   Росинка  station            NaN
5  Залізничний вокзал Луцьк      NaN  train_station


In [16]:
station = stations[(stations['building'] == 'train_station')
                   & stations['name'].fillna('').str.contains('вокзал', case=False)].head(1)
assert len(station) == 1, 'не знайдено головний залізничний вокзал'
print('старт:', station['name'].iloc[0])

старт: Залізничний вокзал Луцьк


In [17]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 9566 вузлів, 26899 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [18]:
def check_snap(points, ids, dist, label):
    ids = np.asarray(ids)
    n, u = len(ids), len(set(ids.tolist()))
    print('%s: точок %d, різних вузлів %d | відстань до вузла: медіана %.1f м, максимум %.1f м'
          % (label, n, u, float(np.median(dist)), float(np.max(dist))))
    if u < n:
        by_node = collections.defaultdict(list)
        for name, nd in zip(points['name'], ids.tolist()):
            by_node[nd].append(name)
        print('  УВАГА: точки, що злиплись на одному вузлі:')
        for nd, names in by_node.items():
            if len(names) > 1:
                print('    вузол %s: %s' % (nd, ' | '.join(names)))
    else:
        print('  ок: кожна точка має власний вузол')
    return u == n

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [19]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    xy = points.to_crs(32635).geometry
    XY = np.c_[xy.x.values, xy.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    mask = np.isfinite(D) & (E > 0)
    np.fill_diagonal(mask, False)
    return float(np.median(D[mask] / E[mask]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [20]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

мала задача: точок 11, різних вузлів 11 | відстань до вузла: медіана 16.0 м, максимум 32.6 м
  ок: кожна точка має власний вузол
матриця за 1.142 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.27


> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [21]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    """Маршрут 0 -> ... -> 0. Щоразу йдемо в найближчу невідвідану точку."""
    k = len(D)
    tour, left = [0], set(range(1, k))
    while left:
        cur = tour[-1]
        nxt = min(left, key=lambda j: D[cur, j])
        tour.append(nxt)
        left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    """Беремо маршрут і розвертаємо ділянки, поки довжина зменшується."""
    best = list(tour)
    best_len = tour_length(D, best)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):
                cand_t = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                L = tour_length(D, cand_t)
                if L < best_len - 1e-9:
                    best, best_len, improved = cand_t, L, True
    return best

In [22]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [23]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [24]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,8176,1.42
2-opt,8062,0.00
Held-Karp (точний),8062,0.00



оптимальний порядок:
   0. СТАРТ: Залізничний вокзал Луцьк
   1. Музей волинської ікони
   2. Кафедральний собор
   3. Костел святих Петра і Павла
   4. Стирова вежа
   5. Художній музей
   6. Владича вежа
   7. Музей книги
   8. Замок Любарта
   9. В'їзна вежа
  10. Волинський краєзнавчий музей
  11. СТАРТ: Залізничний вокзал Луцьк

довжина: 8.06 км


### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [25]:
rows = []
for n in (11, 16, 21, 26, 31):
    cells = 2 ** (n - 1) * (n - 1)
    rows.append({'n (точок)': n, 'клітин у dp': cells,
                 'ГБ (8 байт/клітину)': round(cells * 8 / 1e9, 4)})
growth = pd.DataFrame(rows).set_index('n (точок)')
display(growth)

,клітин у dp,ГБ (8 байт/клітину)
n (точок),,
11,10240,0.0001
16,491520,0.0039
21,20971520,0.1678
26,838860800,6.7109
31,32212254720,257.6980


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [26]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

велика задача: точок 31, різних вузлів 31 | відстань до вузла: медіана 17.9 м, максимум 45.8 м
  ок: кожна точка має власний вузол
коефіцієнт обходу: 1.21
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    13719 м
2-opt            :    13006 м  за 0.050 с  (коротше на 5.2%)

ЦЕ — база порівняння для лабораторної роботи № 8.


---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [27]:
import h3
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

reps = cand_all.geometry.representative_point()      # усі кандидати до очищення дублікатів
cand_all['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in reps]
dens = cand_all['h3'].value_counts()
hexes = gpd.GeoDataFrame(
    {'cell': dens.index, 'n': dens.values},
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens.index],
    crs='EPSG:4326')

cum = dens.cumsum() / dens.sum()
n_half = int((cum < 0.5).sum() + 1)
print('кандидатів: %d | зайнятих комірок: %d' % (len(cand_all), len(dens)))
print('половина всіх пам\u2019яток (%d) лежить у %d комірках (%.0f%% зайнятих)'
      % (len(cand_all) / 2, n_half, 100 * n_half / len(dens)))
print('максимум в одній комірці:', int(dens.max()))

centre = boundary.geometry.iloc[0].centroid
m3 = folium.Map(location=[centre.y, centre.x], zoom_start=12, tiles=BASEMAP)
folium.Choropleth(geo_data=hexes.to_json(), data=hexes, columns=['cell', 'n'],
                  key_on='feature.properties.cell', fill_color='YlOrRd',
                  fill_opacity=0.65, line_opacity=0.3,
                  legend_name='пам\u2019яток у комірці H3 (рівень 9)').add_to(m3)
folium.GeoJson(boundary.to_json(), style_function=lambda f: {
    'color': '#1f4e79', 'weight': 2, 'fill': False}).add_to(m3)
for i in range(len(pts10)):
    folium.CircleMarker([pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i]], radius=5,
                        color='#0b6623', fill=True, fill_opacity=1,
                        tooltip=str(pts10['name'].iloc[i])).add_to(m3)
m3.save('lr5_h3.html')
m3

середнє ребро комірки: 201 м
кандидатів: 140 | зайнятих комірок: 44
половина всіх пам’яток (70) лежить у 7 комірках (16% зайнятих)
максимум в одній комірці: 19


> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [28]:
am = amenities[['name', 'amenity', 'geometry']].rename(columns={'name': 'заклад', 'amenity': 'тип'})
am_m = am.to_crs(32635)
pm = pts10.to_crs(32635)
j = gpd.sjoin_nearest(pm, am_m, how='left', distance_col='відстань_м')
j = j[~j.index.duplicated(keep='first')]          # при рівній віддалі лишаємо один
display(pd.DataFrame({'точка маршруту': j['name'], 'найближчий заклад': j['заклад'],
                      'тип': j['тип'], 'відстань, м': j['відстань_м'].round(0)})
        .reset_index(drop=True))
print('медіана відстані до закладу: %.0f м' % j['відстань_м'].median())

,точка маршруту,найближчий заклад,тип,"відстань, м"
0,СТАРТ: Залізничний вокзал Луцьк,NaN,toilets,25.0
1,Художній музей,NaN,toilets,46.0
2,Волинський краєзнавчий музей,Капучіно,cafe,193.0
3,Музей волинської ікони,П'ятниця,cafe,200.0
4,Музей книги,NaN,toilets,75.0
5,Замок Любарта,NaN,toilets,49.0
6,В'їзна вежа,Корона Вітовта,restaurant,39.0
7,Стирова вежа,NaN,toilets,17.0
8,Владича вежа,NaN,toilets,100.0
9,Костел святих Петра і Павла,"Навчальний ресторан ""Біля замку""",cafe,67.0


медіана відстані до закладу: 49 м


---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [29]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords

road_ll = route_coords(ex10, ids10)
straight_ll = [(pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i]) for i in ex10]
road30_ll = route_coords(op30, ids30)

fmap = folium.Map(location=[pts10.geometry.y.mean(), pts10.geometry.x.mean()],
                  zoom_start=14, tiles=BASEMAP)
g_road = folium.FeatureGroup('маршрут по ребрах графа (оптимум, 10 об\u2019єктів)').add_to(fmap)
g_str = folium.FeatureGroup('прямі відрізки між точками').add_to(fmap)
g_30 = folium.FeatureGroup('2-opt, 30 об\u2019єктів', show=False).add_to(fmap)
folium.PolyLine(road_ll, color='#1f77b4', weight=5, opacity=0.9,
                tooltip='по ребрах графа').add_to(g_road)
folium.PolyLine(straight_ll, color='#d62728', weight=3, opacity=0.9, dash_array='8 8',
                tooltip='прямі відрізки').add_to(g_str)
folium.PolyLine(road30_ll, color='#2ca02c', weight=3, opacity=0.8).add_to(g_30)
for step, i in enumerate(ex10[:-1]):
    name = str(pts10['name'].iloc[i])
    lat, lon = pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i]
    color = '#0b6623' if step == 0 else '#1f4e79'
    folium.Marker([lat, lon], tooltip='%d. %s' % (step, name),
                  icon=folium.DivIcon(html=(
                      '<div style="background:%s;color:#fff;border-radius:50%%;width:24px;'
                      'height:24px;line-height:24px;text-align:center;font-weight:bold;'
                      'border:2px solid #fff">%s</div>' % (color, 'S' if step == 0 else step)),
                      icon_size=(24, 24), icon_anchor=(12, 12))).add_to(fmap)
folium.LayerControl().add_to(fmap)
fmap.save('lr5_маршрут.html')

L_straight = sum(float(E10[a, b]) for a, b in zip(ex10[:-1], ex10[1:]))
print('маршрут з 10 об\u2019єктів: по прямих %.2f км, дорогою %.2f км (відношення %.2f)'
      % (L_straight / 1000, L_opt / 1000, L_opt / L_straight))
print('карту збережено: lr5_маршрут.html')
fmap

маршрут з 10 об’єктів: по прямих 6.64 км, дорогою 8.06 км (відношення 1.21)
карту збережено: lr5_маршрут.html


---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [30]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (16.7 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [31]:
REPORT = f'''
Варіант 10, місто Луцьк (за списком CITIES; вручну не змінювалось).
Дані: OpenStreetMap, живий запит до overpass-api.de (з двох адрес відповідала одна,
закріплена 65.109.112.52), резервний знімок викладача не використовувався.

1. Відкинуто 63 об'єкти зі 203 у шарі heritage. Правило одне: немає тега name
   (теги historic / tourism підходили всім 203, бо запит уже їх відбирає).
   Серед відкинутих: 34 historic=building, 9 citywalls, 8 historic=yes, 5 memorial,
   6 tourism=artwork/viewpoint, 1 factory. Лишилось 140 кандидатів (98 Point, 42 Polygon).
   Без пояснень цей фільтр замовнику не віддав би: безіменний об'єкт не можна показати
   туристу, але частина відкинутих (напр. 9 фрагментів міських мурів) історично цінна,
   просто погано підписана в OSM. Це втрата даних, а не втрата цінності.

2. Рейтинг = 0.35*wiki + 0.25*lang_pct + 0.20*visit + 0.15*hist + 0.05*area_pct.
   wiki: є wikidata або wikipedia (44 із 140). lang_pct: перцентиль числа колонок name:*
   (максимум 14 мов у Замку Любарта). visit: тип tourism (museum, gallery = 1; attraction 0.8;
   viewpoint 0.6; artwork 0.3). hist: тип historic (castle, citywalls, tower, ruins = 1;
   monument, memorial = 0.5; решта 0.6). area_pct: площа, мінімальна вага, бо великою буває й
   споруда без цінності. Найбільші ваги мають сигнали, які ставить стороння людина
   (стаття, переклади): їх складніше «накрутити», ніж тег. Ваги не оптимізувались,
   це експертна оцінка; їхня чутливість не перевірялась.

3. Збіглось 3 з 10 (Замок Любарта, Костел Петра і Павла, Кафедральний собор).
   Площа обирає великі комплекси (Окільний замок, три монастирі, корпус ВНУ, військова техніка),
   мій критерій обирає музеї й вежі з Вікіданими. Обидва списки отримані з тих самих даних
   без помилок, але розходяться на 7 з 10, тож слово «найкращі» не має сенсу без критерію:
   це наслідок обраних ваг, а не властивість міста.

4. Коефіцієнт обходу: 1.27 для 10 об'єктів і 1.21 для 30 (у межах типових 1.2–1.6, не 1.0 і
   не понад 2). Недосяжних пар немає (inf = 0). Асиметрія матриці 0 м: пішохідна мережа
   симетрична, односторонніх відрізків немає. Усі точки мають власний вузол графа
   (11 з 11 і 31 з 31, максимальна відстань до вузла 32.6 м і 45.8 м). На 8.06 км маршруту
   прямі відрізки дали б лише 6.64 км, тож «по прямій» занижує довжину на ~18%.

5. Найближчий сусід гірший за точний розв'язок на 1.42% (8176 м проти 8062 м); 2-opt на цій
   задачі вийшов точно в оптимум (0.00%). Не здивувало: для 10 точок у щільному центрі
   евристика майже не помиляється. На 30 точках 2-opt скоротив маршрут найближчого сусіда
   з 13719 м до 13006 м (на 5.2%) за {t_op30:.3f} с, а скільки він програє оптимуму там, невідомо:
   точний розв'язок недоступний.

6. Заважає розмір таблиці динамічного програмування Held-Karp: 2^(n-1)*(n-1) клітин.
   Для n = 31 це 32 212 254 720 клітин, при 8 байтах 257.7 ГБ (ще стільки ж на таблицю
   батьків), тобто пам'ять, а не час. Для n = 26 це 6.7 ГБ, для 21 лише 0.17 ГБ:
   на кожні 5 нових точок потрібно у 32 рази більше пам'яті. Час теж росте як 2^n * n^2.

7. Що виявилось неправильним у даних:
   - 63 об'єкти без назви (див. п. 1).
   - Дублікати: знайдено 12 пар ближче 60 м серед перших 30. Вилучено 7 записів, де вузол лежить
     усередині малої будівлі (< 3000 м²) іншого об'єкта або збігаються назви: Шляхетський
     будинок, Будинок родини Косачів, Луцькі підземелля, Музей дзвонів, «В пам'ять євреїв
     Луцька», Монастир василіан, Доповнена реальність. Спочатку правило «контури перетинаються»
     вилучило Замок Любарта (у його стінах стоять музеї), тож я обмежив правило малими
     будівлями: музеї в стінах замку це різні об'єкти.
   - Слабкі сигнали: усі 30 лідерів мають wiki = 1, тож цей сигнал уже не
     розрізняє їх, а 5 скульптур «Кликун ...» у верхній тридцятці мають однаковий score 0.546.
   - Заклади (amenities): для 6 із 11 точок маршруту найближчий заклад не має назви
     (усі шість — туалети), медіана відстані до закладу 49 м.
   - Висновок, якого не було в завданні: топ-10 за моїм критерієм складається переважно з
     музеїв (4 перші місця), бо вага visit дорівнює 0.20, хоча Замок Любарта, найвідоміший
     об'єкт, лише п'ятий. Рейтинг відображає зміщення тегування OSM (музеї краще описані),
     а не «цінність». Живий запит коштував 296 с і 5.36 МБ, знімок {t_snap:.2f} с (у {t_live / max(t_snap, 1e-9):.0f} разів швидше в цьому запуску;
     час читання знімка коливається між запусками, мережею передано 0 МБ).

Декларація про використання ШІ:
Код завдань (fetch, фільтр, рейтинг, дублікати, check_snap, матриці, найближчий сусід, 2-opt,
H3, sjoin_nearest, карта) і чернетку звіту складав ШІ-помічник Claude Code за шаблоном
викладача. Перевірено: виконано ноутбук повністю; пройшли вбудовані перевірки
(check_tour, 2-opt не гірший за вхідний, точний не гірший за наближений); різні вузли = точки;
коефіцієнт обходу і асиметрія звірені з контрольними ознаками; помилку правила дублікатів
(втрата Замку Любарта) знайдено за виводом і виправлено. Ваги рейтингу експертні, не перевірені
на чутливість; студент має самостійно прочитати код і числа перед здачею.

Дані © OpenStreetMap contributors, ліцензія ODbL (openstreetmap.org/copyright).
'''
print(REPORT)


Варіант 10, місто Луцьк (за списком CITIES; вручну не змінювалось).
Дані: OpenStreetMap, живий запит до overpass-api.de (з двох адрес відповідала одна,
закріплена 65.109.112.52), резервний знімок викладача не використовувався.

1. Відкинуто 63 об'єкти зі 203 у шарі heritage. Правило одне: немає тега name
   (теги historic / tourism підходили всім 203, бо запит уже їх відбирає).
   Серед відкинутих: 34 historic=building, 9 citywalls, 8 historic=yes, 5 memorial,
   6 tourism=artwork/viewpoint, 1 factory. Лишилось 140 кандидатів (98 Point, 42 Polygon).
   Без пояснень цей фільтр замовнику не віддав би: безіменний об'єкт не можна показати
   туристу, але частина відкинутих (напр. 9 фрагментів міських мурів) історично цінна,
   просто погано підписана в OSM. Це втрата даних, а не втрата цінності.

2. Рейтинг = 0.35*wiki + 0.25*lang_pct + 0.20*visit + 0.15*hist + 0.05*area_pct.
   wiki: є wikidata або wikipedia (44 із 140). lang_pct: перцентиль числа колонок name:*
   (максимум 14 мов у Зам